In [1]:
from openai import OpenAI
import gradio as gr
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
from enum import Enum

c:\Users\LENOVO\Documents\projects\llm_mini_projects\prisoners-dilemma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Defining local ollama variables

DEEPSEEK_R1_1_5B = "deepseek-r1:1.5b"
LLAMA_3_2_1B = "llama3.2:1b"
GEMMA_3_270M = "gemma3:270m"

In [3]:
# Validate ollama running
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [4]:
# Ollama instance
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [5]:
# Defining the game for 3 agents (not specifically meaning AI Agents - Agents as in anything that can perceive the environment and act on it)

PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT = """
## Prisoner's Dilemma for three suspects

The 3-player Prisoner's Dilemma extends the classic two-player game to three suspects - Adam, Eve and Bob who must independently choose whether to Cooperate (stay silent) or Defect (confess / betray).


## The Setup & Rules

Each player acts in self-interest without knowing the others' choices. Based on a standard jail-time payoff structure (where fewer months in prison is better):
• All three cooperate (CCC): Each player gets 4 months in jail.
• All three defect (DDD): Each player gets 8 months in jail.
• Two defect, one cooperates (e.g., DDC, DCD, CDD):
	• The two defectors get 6 months each.
	• The single cooperator gets 12 months.
• One defects, two cooperate (e.g., CCD, CDC, DCC):
	• The single defector walks free (0 months).
	• The two cooperators get 12 months each.

You are a part of the set of 3 players playing this game. Your goal is to minimize your own jail time.    

Your response should be in a clear markdown format without unnecessary backticks. 

The number of total rounds the same three players will play is 5. Each round begins fresh, but with each player aware of the response made by the other 2. This response generated should be considered by you for your next response - deciding the other player(s) strategy and intent. 

The response should be in the following format - 
reason : Reason for choice
choice : The choice itself

reason should be short, brief, and crisp. Number of words generated not more than 20. 
choice can be either "Stay Silent" or "Betray" ONLY. 

When reasoning, respond in first person. 
"""

In [6]:
adam_messages = ["Hi, I'm Adam !"]
eve_messages = ["Hi, I'm Eve !"]
bob_messages = ["Hi, I'm Bob !"]

In [7]:
class Choice(str, Enum):
    STAY_SILENT = "Stay Silent"
    BETRAY = "Betray"

class DilemmaMessageResponseFormat(BaseModel):
    reason: str = Field(description="Reason for choice")
    choice: Choice = Field(description="The choice")

In [8]:
ACTION_MAP = {
    Choice.STAY_SILENT: 'C',
    Choice.BETRAY: 'D'
}

JAIL_PATTERNS = {
    "CCC": (4, 4, 4),
    "CCD": (12, 12, 0), "CDC": (12, 0, 12), "DCC": (0, 12, 12),
    "DDC": (6, 6, 12), "DCD": (6, 12, 6), "CDD": (12, 6, 6),
    "DDD": (8, 8, 8)
}

hours_in_jail = [0, 0, 0]

def update_score(adam_choice, eve_choice, bob_choice, hours_in_jail):
    combination = ACTION_MAP[adam_choice] + ACTION_MAP[eve_choice] + ACTION_MAP[bob_choice]
    current_hours = JAIL_PATTERNS[combination]
    hours_in_jail[:] = [current_hour + new_hour for current_hour, new_hour in zip(hours_in_jail, current_hours)]

In [9]:
PLAYERS = {"Adam": DEEPSEEK_R1_1_5B, "Eve": LLAMA_3_2_1B, "Bob": GEMMA_3_270M}
history = []  # list of {"Adam": Choice, "Eve": Choice, "Bob": Choice}

def call_player(name, round_no):
    others = [n for n in PLAYERS if n != name]
    system = (PRISONERS_DILEMMA_TRIO_PART_SYSTEM_PROMPT
              + f"\n\nYou are {name}. The other players are {others[0]} and {others[1]}.")
    past = "\n".join(
        f"Round {i+1}: " + ", ".join(f"{n} chose {c.value}" for n, c in r.items())
        for i, r in enumerate(history)
    ) or "No rounds played yet."
    user = f"{past}\n\nThis is round {round_no} of 5. Make your choice now."

    response = ollama.beta.chat.completions.parse(
        model=PLAYERS[name],
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        response_format=DilemmaMessageResponseFormat,
        temperature=0.7,
        max_completion_tokens=500,
    )
    parsed = response.choices[0].message.parsed
    if parsed is None:
        raise ValueError(f"{name}: could not parse response")
    return parsed

In [ ]:
for round_no in range(1, 6):
    results = {n: call_player(n, round_no) for n in PLAYERS} 
    print(f"Round: {round_no}")
    for n, r in results.items():
        print(f"{n}: {r.choice.value} - {r.reason}")
    print("*"*50)
    update_score(results["Adam"].choice, results["Eve"].choice, results["Bob"].choice, hours_in_jail)
    history.append({n: r.choice for n, r in results.items()})

Round: 1
Adam: Stay Silent - Staying ensures that if both Eve and Bob stay silent, you receive 4 months. If either of them bet, you receive 8 months, which is worse than 4 months. Therefore, staying is better in all scenarios.
Eve: Stay Silent - I will Cooperate
Bob: Stay Silent - Stay Silent
**************************************************
Round: 2
Adam: Betray - By betraying, Adam can guarantee a better outcome overall. If both Eve and Bob also choose to betray, Adam is free while both receive 12 months. However, if Adam betrays while Eve and Bob stay silent, Adam walks free, and Eve and Bob each receive 12 months. Staying silent would lead to Adam getting 4 months, but others staying silent would also result in a total of 8 months for all three, which is worse than the 24 months if Adam betrays and others also stay silent. Therefore, betraying is the optimal choice for Adam to minimize his own jail time.
Eve: Stay Silent - Adam and Bob are confident in my innocence, so I'll bet an